# 第3回A 演習: 小さな言語モデルを動かす

スライド: [第3回A 小さな言語モデルを手元で動かす](../03a-llm/)

## 準備
**Colab で `ランタイム → ランタイムのタイプを変更 → T4 GPU` にしておくこと。**
CPU でも動くが、生成に数十秒かかる。

モデルのダウンロードに数分かかる。最初のセルを実行したら待つこと。

In [ ]:
!pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(device, '/ 語彙数:', len(tok))

In [ ]:
# 以降で使う小さなヘルパ: 文字列 -> トークン ID のテンソル
def encode(text):
    return tok(text, return_tensors='pt').input_ids.to(device)


# 現在の系列から「次のトークンの logits」を取り出す
def next_logits(ids):
    with torch.no_grad():
        return model(ids).logits[0, -1]


print(encode('こんにちは').shape, next_logits(encode('こんにちは')).shape)

---
## 課題1: 次のトークンの候補を見る

文字列を受け取り、次に来るトークンの**上位 n 候補**を
`[(トークン文字列, 確率), ...]` の形で返す `top_candidates(text, n=10)` を書け。

**ヒント**: `next_logits` → `torch.softmax(..., dim=-1)` → `torch.topk(probs, n)`。
`topk` の返り値は `.values` と `.indices` を持つ。

In [ ]:
def top_candidates(text, n=10):
    # TODO
    pass


for t, p in top_candidates('むかしむかし、あるところに'):
    print(f'{t!r:20} {p:.4f}')

In [ ]:
# 確認
cand = top_candidates('日本の首都は', n=10)
assert len(cand) == 10
assert all(isinstance(t, str) for t, _ in cand)
probs = [p for _, p in cand]
assert probs == sorted(probs, reverse=True), '確率の降順になっていない'
assert 0 < probs[0] <= 1.0
print('OK')

---
## 課題2: greedy 生成

毎回もっとも確率の高いトークンを選び続ける `generate_greedy(prompt, max_new_tokens)` を書け。

- 終端トークン (`tok.eos_token_id`) が出たら止める
- 返すのは**プロンプトを含めた全体**の文字列

**ヒント**: `torch.argmax(logits)` / `torch.cat([ids, next_id.view(1, 1)], dim=1)`。

In [ ]:
def generate_greedy(prompt, max_new_tokens=40):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        logits = next_logits(ids)
        # TODO
        pass
    return tok.decode(ids[0], skip_special_tokens=True)


print(generate_greedy('むかしむかし、あるところに'))

In [ ]:
# 確認: greedy は決定的なので、2回実行すると必ず同じ結果になる
a = generate_greedy('日本で一番高い山は', 20)
b = generate_greedy('日本で一番高い山は', 20)
assert a == b, 'greedy なのに結果が変わっている'
assert a.startswith('日本で一番高い山は')
assert len(a) > len('日本で一番高い山は'), '何も生成されていない'
print('OK')
print(a)

### 課題2-b: greedy の弱点を観察する

`generate_greedy('好きな食べ物は', 80)` のように長めに生成してみて、
何が起きるか観察し、コメントで書け。

In [ ]:
print(generate_greedy('好きな食べ物は', 80))

# 観察したこと:
# 

---
## 課題3: 温度付き top-k サンプリング

`sample_next(logits, temperature, k)` を書け。

1. `logits / temperature` を softmax して確率にする
2. 上位 `k` 個だけを候補にする
3. その確率に比例して1つ引く

**ヒント**: `torch.multinomial(weights, 1)` が重み付きのくじ引き。
返るのは「上位 k 個の中での位置」なので、元の ID に戻す必要がある。

In [ ]:
def sample_next(logits, temperature=0.8, k=50):
    # TODO
    pass


def generate(prompt, max_new_tokens=60, temperature=0.8, k=50):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        next_id = sample_next(next_logits(ids), temperature, k)
        if next_id.item() == tok.eos_token_id:
            break
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tok.decode(ids[0], skip_special_tokens=True)


print(generate('むかしむかし、あるところに'))

In [ ]:
# 確認
logits = next_logits(encode('日本の首都は'))

# k=1 なら greedy と一致するはず
for _ in range(5):
    assert sample_next(logits, temperature=1.0, k=1).item() == torch.argmax(logits).item()

# 候補は必ず上位 k 個の中から出る
topk_ids = set(torch.topk(logits, 50).indices.tolist())
for _ in range(30):
    assert sample_next(logits, 1.0, 50).item() in topk_ids

# 温度を上げれば、選ばれるトークンの種類が増えるはず
cold = {sample_next(logits, 0.1, 50).item() for _ in range(30)}
hot  = {sample_next(logits, 2.0, 50).item() for _ in range(30)}
assert len(hot) > len(cold), f'低温 {len(cold)} 種 / 高温 {len(hot)} 種'
print('OK')

### 課題3-b: 温度を比べる

同じプロンプトで温度 `0.3 / 0.8 / 1.5` の出力を並べ、違いをコメントで書け。

In [ ]:
prompt = 'プログラミングを学ぶと'
for t in [0.3, 0.8, 1.5]:
    print(f'--- temperature = {t} ---')
    print(generate(prompt, max_new_tokens=50, temperature=t))
    print()

# 気づいたこと:
# 

---
## 課題4: 会話する

`chat(messages)` を書け。

1. `tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` で
   プロンプト文字列を作る
2. `sample_next` を使って生成する
3. **新しく生成した部分だけ**を文字列で返す

**ヒント**: 生成前の `ids.shape[1]` を覚えておき、最後に `ids[0, start:]` を decode する。
停止判定は `next_id.item() in tok.all_special_ids` で行う。

In [ ]:
def chat(messages, max_new_tokens=100, temperature=0.7, k=50):
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    ids = encode(prompt)
    start = ids.shape[1]
    # TODO
    return tok.decode(ids[0, start:], skip_special_tokens=True)


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': '日本の首都はどこですか？'},
]
print(chat(msgs))

In [ ]:
# 確認
msgs = [{'role': 'user', 'content': '1 + 1 は？'}]
reply = chat(msgs, max_new_tokens=30)
assert isinstance(reply, str) and reply.strip(), '空の応答'
assert '1 + 1 は？' not in reply, 'プロンプトが混ざっている (start の位置を見直す)'
assert '<|im_start|>' not in reply, '特殊トークンが残っている'
print('OK')
print(reply)

### 課題4-b: 3往復の会話

下のセルを実行し、**3往復以上**会話してから `quit` で抜けること。
そのやりとりを提出物に残す。

2回目以降で、前の話題を踏まえた応答になるか観察せよ。

In [ ]:
history = [{'role': 'system', 'content': '簡潔に答えてください。'}]

while True:
    user = input('あなた: ')
    if user.strip() in ('quit', 'exit', ''):
        break
    history.append({'role': 'user', 'content': user})
    reply = chat(history)
    print('AI:', reply)
    history.append({'role': 'assistant', 'content': reply})

---
# 発展課題

## 発展1: top-p (nucleus) サンプリング

確率の高い順に足していき、累積が `p` を超える手前までを候補にする
`sample_next_top_p(logits, temperature, p)` を書け。

**ヒント**: `torch.sort(probs, descending=True)` → `torch.cumsum` →
`cumsum - sorted_probs < p` でマスクを作る (最上位が必ず残るように)。

In [ ]:
def sample_next_top_p(logits, temperature=0.8, p=0.9):
    # TODO
    pass

In [ ]:
# 確認
logits = next_logits(encode('日本の首都は'))
probs = torch.softmax(logits, dim=-1)

# p が極小なら、実質 greedy
for _ in range(5):
    assert sample_next_top_p(logits, 1.0, 1e-6).item() == torch.argmax(logits).item()

# p=0.9 の候補は、累積 0.9 に届くまでの語だけ
sp, si = torch.sort(probs, descending=True)
n_keep = int((torch.cumsum(sp, -1) - sp < 0.9).sum())
allowed = set(si[:n_keep].tolist())
for _ in range(30):
    assert sample_next_top_p(logits, 1.0, 0.9).item() in allowed
print(f'OK (候補は {n_keep} 語)')

## 発展2: 繰り返しペナルティ

すでに出たトークンを選びにくくする。
`penalty > 1` のとき、既出トークンの logits を `penalty` で割ってから選ぶ
(負の logits は逆に掛ける、が一般的な実装)。

`generate_penalized(prompt, penalty=1.2)` を書き、
課題2-b で観察した繰り返しが減るか確かめよ。

In [ ]:
def apply_penalty(logits, used_ids, penalty=1.2):
    # TODO: used_ids に含まれる位置の logits を調整して返す
    pass


def generate_penalized(prompt, max_new_tokens=80, penalty=1.2):
    # TODO
    pass


print(generate_penalized('好きな食べ物は'))

---
## 提出

すべてのセルを実行し、**出力とコメントを残したまま** `.ipynb` をダウンロードして提出。
観察を書くよう指示されているセルは、必ずコメントを埋めること。